# Paper 2 — Goal 1 FINAL FREEZE — v1.0

This notebook performs **no modeling and no statistical recomputation**.

It freezes Goal 1 only if all of the following already exist and pass their provenance gates:

- Notebook 10B computational-completion manifest;
- canonical Goal 1 machine-readable outputs;
- finalized diagnosis and severity permutation inference;
- final 2,000-bootstrap metric tables;
- final sensitivity summary;
- final publication-figure package from Notebook 10C FINAL v1.2.

If every gate passes, the notebook writes:

- `GOAL1_FINAL_FREEZE.json`
- `DONE.json`

with `status = "PASS"`.

After this notebook passes, Goal 1 is formally frozen and should not be reopened unless a genuine upstream scientific or provenance error is discovered.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

FREEZE_ENGINE = "goal1-final-freeze-v1.0.0"

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

def atomic_json(payload, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name("." + path.name + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def find_root():
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        p = Path(override).expanduser().resolve()
        if (p / "outputs" / "goal1").exists():
            return p
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {p}")

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "outputs" / "goal1").exists() and (p / "data").exists():
            return p
    raise FileNotFoundError(
        "Could not locate Paper_2_Leakage/Code. Run from the repository "
        "or set PAPER2_ROOT."
    )

ROOT = find_root()
GOAL1_ROOT = ROOT / "outputs" / "goal1"
FINAL = GOAL1_ROOT / "goal1_complete_v1_1" / "final"
TABLES = FINAL / "tables"
COMPLETION = FINAL / "completion_v1_0"
FIGURES = GOAL1_ROOT / "FINAL_FIGURES_V1_2"
FREEZE = FINAL / "final_freeze_v1_0"
FREEZE.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Goal 1 authoritative final:", FINAL)
print("Figure package:", FIGURES)
print("Freeze folder:", FREEZE)

Project root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Goal 1 authoritative final: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final
Figure package: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\FINAL_FIGURES_V1_2
Freeze folder: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final\final_freeze_v1_0


## 1. Computational-completion gate

In [2]:
completion_manifest_path = COMPLETION / "GOAL1_COMPLETION_MANIFEST.json"
if not completion_manifest_path.exists():
    raise FileNotFoundError(f"Missing completion manifest: {completion_manifest_path}")

completion = json.loads(completion_manifest_path.read_text(encoding="utf-8"))

if completion.get("status") != "PASS_PENDING_FIGURE_REVIEW":
    raise RuntimeError(
        "Goal 1 computational completion is not in the expected passed state: "
        f"{completion.get('status')!r}"
    )

required_contract = {
    "primary_outer_folds": 5,
    "primary_outer_repeats": 10,
    "bootstrap_replicates": 2000,
    "formal_permutations_per_task": 1000,
    "formal_permutation_repeats": 3,
    "old_invalid_fixed_manifest_diagnosis_null_used": False,
}
for key, expected in required_contract.items():
    observed = completion.get(key)
    if observed != expected:
        raise RuntimeError(
            f"Completion contract mismatch for {key}: expected {expected!r}, found {observed!r}"
        )

print("GOAL 1 COMPUTATIONAL COMPLETION GATE: PASS")

GOAL 1 COMPUTATIONAL COMPLETION GATE: PASS


## 2. Canonical statistical-output gate

In [3]:
required_tables = [
    TABLES / "goal1_dx_oof.csv",
    TABLES / "goal1_bulbar_oof.csv",
    TABLES / "goal1_metrics.csv",
    TABLES / "goal1_permutation.csv",
    TABLES / "goal1_splits.csv",
    TABLES / "goal1_sensitivity_summary.csv",
    TABLES / "diagnosis_coreq_permutation_null.csv",
    TABLES / "severity_coreq_permutation_null.csv",
    TABLES / "diagnosis_bootstrap_ci.csv",
    TABLES / "severity_bootstrap_ci.csv",
    TABLES / "diagnosis_calibration_statistics.csv",
    TABLES / "diagnosis_calibration_curve.csv",
]

missing = [str(p) for p in required_tables if not p.exists() or p.stat().st_size == 0]
if missing:
    raise RuntimeError("Missing/empty canonical Goal 1 outputs:\n" + "\n".join(missing))

metrics = pd.read_csv(TABLES / "goal1_metrics.csv")
sens = pd.read_csv(TABLES / "goal1_sensitivity_summary.csv")
dx_null = pd.read_csv(TABLES / "diagnosis_coreq_permutation_null.csv")
sev_null = pd.read_csv(TABLES / "severity_coreq_permutation_null.csv")

if int(metrics["bootstrap_replicates"].dropna().min()) != 2000:
    raise RuntimeError("Final metric table is not uniformly based on 2,000 bootstraps.")
if not sens["bootstrap_replicates"].eq(2000).all():
    raise RuntimeError("Sensitivity table is not uniformly based on 2,000 bootstraps.")
if len(dx_null) != 1000 or dx_null["permutation"].nunique() != 1000:
    raise RuntimeError("Diagnosis formal null is not exactly 1,000 unique draws.")
if len(sev_null) != 1000 or sev_null["permutation"].nunique() != 1000:
    raise RuntimeError("Severity formal null is not exactly 1,000 unique draws.")

print("CANONICAL STATISTICAL OUTPUT GATE: PASS")

CANONICAL STATISTICAL OUTPUT GATE: PASS


## 3. Final publication-figure gate

Only the final `*_FINAL` files produced by Notebook 10C FINAL v1.2 are accepted.

In [4]:
figure_stems = [
    "Figure2_Goal1_information_availability_FINAL",
    "FigureS_G1_01_cohort_flow_FINAL",
    "FigureS_G1_02_CoreQ_availability_FINAL",
    "FigureS_G1_03_diagnosis_calibration_FINAL",
    "FigureS_G1_04_bulbar_prediction_residuals_FINAL",
    "FigureS_G1_05_sensitivities_FINAL",
    "FigureS_G1_06_Q_age_descriptive_FINAL",
]

required_figures = []
for stem in figure_stems:
    for ext in ["pdf", "svg", "png"]:
        required_figures.append(FIGURES / f"{stem}.{ext}")

missing = [str(p) for p in required_figures if not p.exists() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(
        "Final Goal 1 figure package is incomplete. Missing/empty files:\n"
        + "\n".join(missing)
    )

figure_manifest_path = FIGURES / "GOAL1_PUBLICATION_FIGURES_MANIFEST_REVIEW.json"
if not figure_manifest_path.exists():
    raise FileNotFoundError(f"Missing final figure manifest: {figure_manifest_path}")

figure_manifest = json.loads(figure_manifest_path.read_text(encoding="utf-8"))
if figure_manifest.get("engine_version") != "goal1-publication-figures-final-v1.2.0":
    raise RuntimeError(
        "Figure package was not produced by the accepted FINAL v1.2 figure notebook."
    )
if figure_manifest.get("status") != "PASS_PENDING_FINAL_VISUAL_CONFIRMATION":
    raise RuntimeError(
        "Figure package is not in the expected final-review state: "
        f"{figure_manifest.get('status')!r}"
    )

print("FINAL PUBLICATION FIGURE GATE: PASS")

FINAL PUBLICATION FIGURE GATE: PASS


## 4. Final numerical snapshot

This cell records the primary Goal 1 results that are being frozen.

In [5]:
def one_metric(task, model, metric):
    z = metrics.loc[
        metrics["task"].eq(task)
        & metrics["model"].eq(model)
        & metrics["metric"].eq(metric)
    ]
    if len(z) != 1:
        raise RuntimeError(f"Expected one row for {task}/{model}/{metric}; found {len(z)}")
    return z.iloc[0]

dx_core = one_metric("diagnosis", "Core-Q", "AUROC")
dx_age = one_metric("diagnosis", "Age", "AUROC")
dx_ageq = one_metric("diagnosis", "Age + Core-Q", "AUROC")
sev_core = one_metric("severity", "Core-Q", "MAE")
sev_base = one_metric("severity", "Mean baseline", "MAE")
sev_ageq = one_metric("severity", "Age + Core-Q", "MAE")

dx_p = float(pd.to_numeric(dx_null["empirical_p"], errors="raise").iloc[0])
sev_p = float(pd.to_numeric(sev_null["empirical_p"], errors="raise").iloc[0])

snapshot = {
    "diagnosis": {
        "Core-Q_AUROC": float(dx_core["estimate"]),
        "Core-Q_CI95": [float(dx_core["ci_low"]), float(dx_core["ci_high"])],
        "Age_AUROC": float(dx_age["estimate"]),
        "Age+Core-Q_AUROC": float(dx_ageq["estimate"]),
        "permutation_empirical_p": dx_p,
    },
    "severity": {
        "Core-Q_MAE": float(sev_core["estimate"]),
        "Core-Q_CI95": [float(sev_core["ci_low"]), float(sev_core["ci_high"])],
        "Mean_baseline_MAE": float(sev_base["estimate"]),
        "Age+Core-Q_MAE": float(sev_ageq["estimate"]),
        "permutation_empirical_p": sev_p,
    },
}

print(json.dumps(snapshot, indent=2))

{
  "diagnosis": {
    "Core-Q_AUROC": 0.7430379746835443,
    "Core-Q_CI95": [
      0.6750670817688809,
      0.8072264884680851
    ],
    "Age_AUROC": 0.7407301636307503,
    "Age+Core-Q_AUROC": 0.8726921889472059,
    "permutation_empirical_p": 0.0009990009990009
  },
  "severity": {
    "Core-Q_MAE": 1.685234566140279,
    "Core-Q_CI95": [
      1.4774227442576435,
      1.898178407728193
    ],
    "Mean_baseline_MAE": 2.018413572716013,
    "Age+Core-Q_MAE": 1.6610477017359702,
    "permutation_empirical_p": 0.0009990009990009
  }
}


## 5. Write FINAL Goal 1 freeze/seal

In [6]:
all_frozen_artifacts = [
    completion_manifest_path,
    figure_manifest_path,
    *required_tables,
    *required_figures,
    COMPLETION / "audit" / "goal1_permutation_method_amendment.json",
    COMPLETION / "tables" / "goal1_permutation_final_summary.csv",
]

missing = [str(p) for p in all_frozen_artifacts if not p.exists() or p.stat().st_size == 0]
if missing:
    raise RuntimeError(
        "Cannot freeze Goal 1; required artifact missing/empty:\n" + "\n".join(missing)
    )

artifact_hashes = {
    str(p.relative_to(ROOT)): sha256_file(p)
    for p in all_frozen_artifacts
}

freeze_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "goal": 1,
    "freeze_engine": FREEZE_ENGINE,
    "scientific_question": (
        "Does the recording-quality representation Q contain reproducible clinical "
        "information in unseen participants?"
    ),
    "primary_analysis": {
        "outer_folds": 5,
        "outer_repeats": 10,
        "inner_folds": 5,
        "participant_grouped": True,
    },
    "uncertainty": {
        "participant_bootstrap_replicates": 2000,
    },
    "formal_permutation_inference": {
        "permutations_per_task": 1000,
        "matched_outer_repeats": 3,
        "diagnosis_regenerated_stratification_after_each_permutation": True,
        "severity_fixed_outcome_independent_master_folds": True,
        "old_invalid_fixed_manifest_diagnosis_null_used": False,
    },
    "primary_results": snapshot,
    "figure_package": {
        "directory": str(FIGURES.relative_to(ROOT)),
        "main_figure": figure_stems[0],
        "goal1_supplement_internal_ids": figure_stems[1:],
        "global_supplement_numbering": "DEFERRED_UNTIL_MANUSCRIPT_ASSEMBLY",
    },
    "interpretation_boundary": (
        "Goal 1 establishes that recording-quality characteristics contain reproducible "
        "clinical information in unseen participants. It does not establish technical origin, "
        "causal acquisition effects, confounding, spuriousness, or shortcut learning."
    ),
    "frozen_artifact_hashes": artifact_hashes,
    "reopen_policy": (
        "Do not alter Goal 1 analysis, resampling, model definitions, or figures after this "
        "freeze unless a genuine upstream scientific/provenance error is discovered."
    ),
}

freeze_manifest_path = FREEZE / "GOAL1_FINAL_FREEZE.json"
done_path = FINAL / "DONE.json"

atomic_json(freeze_manifest, freeze_manifest_path)
atomic_json(
    {
        "status": "PASS",
        "goal": 1,
        "freeze_manifest": str(freeze_manifest_path.relative_to(ROOT)),
        "freeze_manifest_sha256": sha256_file(freeze_manifest_path),
        "message": "Goal 1 is complete and frozen.",
    },
    done_path,
)

print("=" * 78)
print("GOAL 1 FINAL FREEZE: PASS")
print("=" * 78)
print("Freeze manifest:", freeze_manifest_path)
print("DONE seal:", done_path)
print()
print("GOAL 1 IS NOW COMPLETE AND FROZEN.")

GOAL 1 FINAL FREEZE: PASS
Freeze manifest: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final\final_freeze_v1_0\GOAL1_FINAL_FREEZE.json
DONE seal: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final\DONE.json

GOAL 1 IS NOW COMPLETE AND FROZEN.
